# Loan Default Risk with an Absorbing Markov Chain

**Modelling loan default risk with an absorbing Markov chain in Python.**

## 1. Project overview

A bank's loan moves between a few repayment states each month. A Markov chain says the next state depends only on the current state, so the whole system is described by one table of probabilities, the **transition matrix** $P$.

This project does two things:

1. **Simulation:** start from a transition matrix we know, and generate thousands of synthetic loan histories.
2. **Estimation:** forget the matrix, estimate it from the histories, and compare it with the true one. We also study how the error shrinks as the number of loans grows.

Once the estimated matrix is validated, we use it to compute:

- the probability a loan has defaulted within $n$ months (matrix powers $P^n$),
- the expected number of months until a loan defaults or is repaid (fundamental matrix),
- the probability a loan *eventually* ends in default rather than repayment (absorption probabilities).

> All data and probabilities here are hypothetical. They are not based on real bank records.

## 2. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
rng = np.random.default_rng(SEED)

plt.rcParams["figure.dpi"] = 110

## 3. States and the true transition matrix

There are five states, observed once a month:

| Symbol | State | Type |
|---|---|---|
| C | Current (paying on time) | transient |
| L | Late (missed a payment) | transient |
| V | Very late (missed several payments) | transient |
| D | Default (bank treats the loan as lost) | **absorbing** |
| F | Paid off (loan fully repaid) | **absorbing** |

Default and Paid off are kept separate because they are different endings: one is a loss for the bank, the other is a success.

The **Markov assumption**: the next state depends only on the current state, not on the loan's earlier history.

The true matrix is (rows = current state, columns = next state):

$$
P =
\begin{pmatrix}
0.85 & 0.10 & 0.02 & 0.01 & 0.02 \\
0.20 & 0.50 & 0.20 & 0.05 & 0.05 \\
0.05 & 0.20 & 0.40 & 0.25 & 0.10 \\
0 & 0 & 0 & 1 & 0 \\
0 & 0 & 0 & 0 & 1
\end{pmatrix}
$$

In [ ]:
states = ["C", "L", "V", "D", "F"]
n_states = len(states)

P = np.array([
    [0.85, 0.10, 0.02, 0.01, 0.02],
    [0.20, 0.50, 0.20, 0.05, 0.05],
    [0.05, 0.20, 0.40, 0.25, 0.10],
    [0.00, 0.00, 0.00, 1.00, 0.00],
    [0.00, 0.00, 0.00, 0.00, 1.00],
])

# Every row of a transition matrix must sum to 1
assert np.allclose(P.sum(axis=1), 1), "Rows of P must sum to 1"

pd.DataFrame(P, index=states, columns=states)

## 4. Simulating loan histories

For each loan, at each month we look at its current state, take the matching row of $P$, and draw the next state from that row.

To make this fast, we avoid looping over loans. For all loans at once we draw a uniform random number $u \in [0,1)$ and compare it with the **cumulative** row of $P$. For example, from state C the cumulative row is $(0.85, 0.95, 0.97, 0.98, 1.00)$, so $u = 0.91$ lands in the second state (L).

All loans start in **Current**. Because D and F have a 1 on the diagonal, loans that reach them stay there.

In [ ]:
def simulate_paths(P, n_loans, n_months, rng, start_state=0):
    '''
    Simulate monthly state histories for many loans at once.

    Returns an integer array of shape (n_loans, n_months + 1);
    entry [i, t] is the state index of loan i at month t.
    '''
    cum_P = np.cumsum(P, axis=1)
    cum_P[:, -1] = 1.0                      # guard against rounding error

    paths = np.empty((n_loans, n_months + 1), dtype=int)
    paths[:, 0] = start_state

    for t in range(1, n_months + 1):
        u = rng.random(n_loans)
        # next state = number of cumulative thresholds that u exceeds
        paths[:, t] = (u[:, None] > cum_P[paths[:, t - 1]]).sum(axis=1)

    return paths


n_loans = 10_000
n_months = 36

paths = simulate_paths(P, n_loans, n_months, rng)

# Show the first few loans as readable letters
pd.DataFrame(
    np.array(states)[paths[:5, :13]],
    columns=[f"month_{t}" for t in range(13)],
    index=[f"loan_{i+1}" for i in range(5)],
)

### Sanity check

If the simulation is correct, the share of loans in each state after 36 months should be close to what theory predicts, $\pi_0 P^{36}$, where $\pi_0$ is the starting distribution (everyone in C).

In [ ]:
pi0 = np.zeros(n_states)
pi0[0] = 1.0                                 # all loans start in Current

theory = pi0 @ np.linalg.matrix_power(P, n_months)
observed = np.bincount(paths[:, n_months], minlength=n_states) / n_loans

pd.DataFrame(
    {"Theory (pi0 P^36)": theory, "Simulated": observed},
    index=states,
).round(4)

## 5. Estimating the transition matrix from the data

Now we pretend we do not know $P$. From the histories we count how often each move $i \to j$ happened, and divide by the total number of moves out of state $i$:

$$
\hat P_{ij} = \frac{N_{ij}}{\sum_k N_{ik}}
$$

where $N_{ij}$ is the number of observed transitions from $i$ to $j$. This is the **maximum likelihood estimator** for a Markov chain.

In [ ]:
def count_transitions(paths, n_states):
    '''Count all observed i -> j moves across all loans and months.'''
    counts = np.zeros((n_states, n_states), dtype=int)
    current = paths[:, :-1].ravel()
    nxt = paths[:, 1:].ravel()
    np.add.at(counts, (current, nxt), 1)
    return counts


def estimate_P(paths, n_states):
    '''Maximum likelihood estimate of the transition matrix.'''
    counts = count_transitions(paths, n_states)
    row_totals = counts.sum(axis=1, keepdims=True)
    # rows with no observations are left as zeros instead of dividing by 0
    return np.divide(counts, row_totals,
                     out=np.zeros_like(counts, dtype=float),
                     where=row_totals > 0)


counts = count_transitions(paths, n_states)
P_hat = estimate_P(paths, n_states)

print("Transition counts N_ij:")
display(pd.DataFrame(counts, index=states, columns=states))

print("\nEstimated matrix P_hat:")
display(pd.DataFrame(P_hat, index=states, columns=states).round(4))

print("\nDifference (P_hat - P):")
display(pd.DataFrame(P_hat - P, index=states, columns=states).round(4))

print("\nLargest absolute error:", np.abs(P_hat - P).max().round(4))

### How the error shrinks with more loans

We repeat the estimation with 100, 1,000, 10,000 and 100,000 loans, 20 times each, and record the largest absolute error in $\hat P$. Statistical theory suggests the error should shrink like $1/\sqrt{n}$, so a dashed reference line of that shape is added.

In [ ]:
sample_sizes = [100, 1_000, 10_000, 100_000]
n_repeats = 20

mean_errors = []
for n in sample_sizes:
    errs = []
    for _ in range(n_repeats):
        sim = simulate_paths(P, n, n_months, rng)
        errs.append(np.abs(estimate_P(sim, n_states) - P).max())
    mean_errors.append(np.mean(errs))

error_table = pd.DataFrame({
    "Number of loans": sample_sizes,
    "Mean max abs error": np.round(mean_errors, 5),
})
display(error_table)

# Reference line proportional to 1/sqrt(n), scaled to pass through the first point
ref = mean_errors[0] * np.sqrt(sample_sizes[0] / np.array(sample_sizes))

plt.figure(figsize=(7, 4.5))
plt.loglog(sample_sizes, mean_errors, "o-", label="Observed error")
plt.loglog(sample_sizes, ref, "--", label=r"$1/\sqrt{n}$ reference")
plt.xlabel("Number of loans (log scale)")
plt.ylabel("Largest absolute error in P_hat (log scale)")
plt.title("Estimation error decreases as the sample grows")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 6. Multi-period default probabilities

If a loan starts in state $i$, the probability it is in state $j$ after $n$ months is the $(i,j)$ entry of $P^n$. Since all our loans start in Current, the **cumulative default probability** by month $n$ is $(P^n)_{C,D}$.

We compute it from both the true $P$ and the estimated $\hat P$. Because D is absorbing, this probability can only go up with time.

In [ ]:
def default_curve(M, start=0, default=3, horizon=36):
    '''Probability of having defaulted by each month, for a loan starting in `start`.'''
    pi = np.zeros(M.shape[0])
    pi[start] = 1.0
    curve = [pi[default]]
    for _ in range(horizon):
        pi = pi @ M
        curve.append(pi[default])
    return np.array(curve)


curve_true = default_curve(P)
curve_hat = default_curve(P_hat)

plt.figure(figsize=(8, 4.5))
plt.plot(curve_true * 100, label="True P")
plt.plot(curve_hat * 100, "--", label="Estimated P_hat")
plt.xlabel("Month")
plt.ylabel("Cumulative default probability (%)")
plt.title("Probability a Current loan has defaulted by month n")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

pd.DataFrame(
    {"True P (%)": curve_true[[6, 12, 24, 36]] * 100,
     "Estimated P_hat (%)": curve_hat[[6, 12, 24, 36]] * 100},
    index=["6 months", "12 months", "24 months", "36 months"],
).round(2)

## 7. Expected time and final outcome (absorbing chain theory)

Reorder the matrix so the transient states (C, L, V) come first:

$$
P = \begin{pmatrix} Q & R \\ 0 & I \end{pmatrix}
$$

- $Q$ is the $3\times3$ block of moves among C, L, V.
- $R$ is the $3\times2$ block of moves from C, L, V into D and F.

Then:

- **Fundamental matrix** $N = (I - Q)^{-1}$. Entry $N_{ij}$ is the expected number of months a loan starting in $i$ spends in state $j$ before absorption.
- **Expected time to absorption** $t = N\mathbf{1}$ (row sums of $N$): the average number of months until the loan either defaults or is paid off.
- **Absorption probabilities** $B = NR$. Entry $B_{ik}$ is the probability that a loan starting in $i$ *eventually* ends in absorbing state $k$.

In [ ]:
def absorbing_analysis(M, n_transient=3):
    Q = M[:n_transient, :n_transient]
    R = M[:n_transient, n_transient:]
    N = np.linalg.inv(np.eye(n_transient) - Q)
    expected_time = N @ np.ones(n_transient)
    B = N @ R
    return N, expected_time, B


transient = states[:3]
absorbing = states[3:]

N_true, t_true, B_true = absorbing_analysis(P)
N_hat, t_hat, B_hat = absorbing_analysis(P_hat)

print("Fundamental matrix N (true P):")
display(pd.DataFrame(N_true, index=transient, columns=transient).round(3))

print("\nExpected months until default or repayment:")
display(pd.DataFrame({"True P": t_true, "Estimated P_hat": t_hat},
                     index=transient).round(2))

print("\nProbability of eventually ending in each absorbing state:")
display(pd.DataFrame(
    np.hstack([B_true, B_hat]),
    index=transient,
    columns=pd.MultiIndex.from_product(
        [["True P", "Estimated P_hat"], ["Default (D)", "Paid off (F)"]]),
).round(4))

**Reading the results.** A loan that is currently on time takes the longest to reach an ending (about 14 months), while a very late loan resolves fastest (about 6 months). A very late loan is also the most likely to end in default (about 65%), though even a current loan ends in default around 54% of the time under this hypothetical matrix. The estimated matrix gives values very close to the true ones, which confirms that the estimation method works.

Note that the long-run default probability for a Current loan (about 54%) is slightly above the 36-month figure (about 50.5%), because the few loans still not absorbed after 36 months still carry some default risk.

## 8. Visualising the transition matrix

In [ ]:
plt.figure(figsize=(6.5, 5))
plt.imshow(P, cmap="Blues", vmin=0, vmax=1)

plt.xticks(range(n_states), states)
plt.yticks(range(n_states), states)
plt.xlabel("Next state")
plt.ylabel("Current state")
plt.title("True transition matrix P")

for i in range(n_states):
    for j in range(n_states):
        colour = "white" if P[i, j] > 0.6 else "black"
        plt.text(j, i, f"{P[i, j]:.2f}", ha="center", va="center", color=colour)

plt.colorbar(label="Transition probability")
plt.tight_layout()
plt.show()

## 9. Limitations

- **Hypothetical data.** The matrix was invented. Real banks estimate it from historical loan data, and real probabilities are not known in advance.
- **The Markov assumption.** In reality, a loan that has been late three times before probably behaves differently from one that has been late once. A first-order chain ignores that history.
- **Constant probabilities over time.** We assume the same $P$ every month. In reality, default risk depends on the loan's age, the economy, interest rates and so on (a time-inhomogeneous chain would be needed).
- **All loans are identical.** Loan size, interest rate and borrower type do not affect the transition probabilities here.
- **Estimation error.** With few loans, rare transitions (such as C to D) are estimated poorly. The sample-size experiment in Section 5 shows how much data is needed.
- **No money involved.** We model default *probability*, not the amount lost. A natural extension is expected loss = probability of default × loan amount × loss fraction.

## 10. Summary

1. Built a five-state absorbing Markov chain for loan repayment.
2. Simulated 10,000 synthetic loans over 36 months using a vectorised method.
3. Recovered the transition matrix from the histories and showed the error falls roughly like $1/\sqrt{n}$.
4. Computed cumulative default probabilities, expected time to absorption and eventual default probabilities, from both the true and estimated matrices.